# Part 1: Bag of Words Representation

## Read in Data from Demo Code

### Import Notable Libraries

In [72]:
import pandas as pd
import numpy as np
import os
import textwrap

In [73]:
data_dir = 'data_readinglevel'
x_train_df = pd.read_csv(os.path.join(data_dir, 'x_train.csv'))
y_train_df = pd.read_csv(os.path.join(data_dir, 'y_train.csv'))

N, n_cols = x_train_df.shape
print("Shape of x_train_df: (%d, %d)" % (N, n_cols))
print("Shape of y_train_df: %s" % str(y_train_df.shape))

Shape of x_train_df: (5557, 32)
Shape of y_train_df: (5557, 5)


### Test Input Data Parsing

In [74]:
tr_text_list = x_train_df['text'].values.tolist()
prng = np.random.RandomState(101)
rows = prng.permutation(np.arange(y_train_df.shape[0]))
for row_id in rows[:8]:
    text = tr_text_list[row_id]
    print("row %5d | %s BY %s | y = %s" % (
        row_id,
        y_train_df['title'].values[row_id],
        y_train_df['author'].values[row_id],
        y_train_df['Coarse Label'].values[row_id],
        ))
    # Pretty print text via textwrap library
    line_list = textwrap.wrap(tr_text_list[row_id],
        width=70,
        initial_indent='  ',
        subsequent_indent='  ')
    print('\n'.join(line_list))
    print("")

row  4746 | The Red and the Black: A Chronicle of 1830 BY Stendhal | y = Key Stage 4-5
  It was hermetically sealed; he was on the point of  fainting and
  remained for a long time leaning against the oak; then  with a
  staggering step he went to have another look at the gardener's
  ladder. The chain which he had once forced asunder--in, alas, such
  different  circumstances--had not yet been repaired. Carried away by
  a moment of  madness, Julien pressed it to his lips.

row  1250 | Cranford BY Elizabeth Cleghorn Gaskell | y = Key Stage 4-5
  Miss Pole, Miss Matty, and I, meanwhile attended to Miss Brown: and
  hard  work we found it to relieve her querulous and never-ending
  complaints. But if we were so weary and dispirited, what must Miss
  Jessie have been! Yet she came back almost calm as if she had gained
  a new strength. She  put off her mourning dress, and came in,
  looking pale and gentle,  thanking us each with a soft long pressure
  of the hand.

row    54 | The Big F

## Part 1: Cleaning CSV File into a list of Tokens

We first define the our tokenize_text function

In [75]:
def tokenize_text(raw_text):
    ''' Transform a plain-text string into a list of tokens
    
    We assume that *whitespace* divides tokens.
    
    Args
    ----
    raw_text : string
    
    Returns
    -------
    list_of_tokens : list of strings
        Each element is one token in the provided text
    '''
    list_of_tokens = raw_text.split() # split method divides on whitespace by default
    for pp in range(len(list_of_tokens)):
        cur_token = list_of_tokens[pp]
        # Remove punctuation
        for punc in ['?', '!', '_', '.', ',', '"', '/']:
            cur_token = cur_token.replace(punc, "")
        # Turn to lower case
        clean_token = cur_token.lower()
        # Replace the cleaned token into the original list
        list_of_tokens[pp] = clean_token
    return list_of_tokens

### Testing our Tokenize Function on the tr_test_list:

In [76]:
for line in tr_text_list[:10]:
    print("\nRaw text:")
    print(line)
    print("Clean token list:")
    print(tokenize_text(line))


Raw text:
"Yes.... What sort of terms was he on with the guests—you and Miss  Norris and all of them?" "Just polite and rather silent, you know. Keeping himself to himself. We didn't see so very much of him, except at meals. We were here to  enjoy ourselves, and—well, _he_ wasn't." "He wasn't there when the ghost walked?" "No. I heard Mark calling for him when he went back to the house. I  expect Cayley stroked down his feathers a bit, and told him that girls  will be girls....—Hallo, here we are."
Clean token list:
['yes', 'what', 'sort', 'of', 'terms', 'was', 'he', 'on', 'with', 'the', 'guests—you', 'and', 'miss', 'norris', 'and', 'all', 'of', 'them', 'just', 'polite', 'and', 'rather', 'silent', 'you', 'know', 'keeping', 'himself', 'to', 'himself', 'we', "didn't", 'see', 'so', 'very', 'much', 'of', 'him', 'except', 'at', 'meals', 'we', 'were', 'here', 'to', 'enjoy', 'ourselves', 'and—well', 'he', "wasn't", 'he', "wasn't", 'there', 'when', 'the', 'ghost', 'walked', 'no', 'i', 'heard'

## Part 2: Defining a Fixed Size Vocabulary 

Firstly, count the number of instances a particular token shows up:

In [77]:
tok_count_dict = dict()

for line in tr_text_list:
    tok_list = tokenize_text(line)
    for tok in tok_list:
        if tok in tok_count_dict:
            tok_count_dict[tok] += 1
        else:
            tok_count_dict[tok] = 1

Print 10 most common tokens:

In [78]:
sorted_tokens = list(sorted(tok_count_dict, key=tok_count_dict.get, reverse=True))

print("Top 10 most frequent")
for w in sorted_tokens[:10]:
    print("%5d %s" % (tok_count_dict[w], w))

print("10 Least Frequent:\n")
for w in sorted_tokens[-10:]:
    print("%5d %s" % (tok_count_dict[w], w))

Top 10 most frequent
24173 the
14483 and
12029 of
11903 to
 9227 a
 6778 in
 6728 i
 5982 he
 5501 that
 5258 was
10 Least Frequent:

    1 honorable;
    1 saucy
    1 brandied
    1 absinthe
    1 teased
    1 coquenard
    1 madinier
    1 hearse
    1 lumpy
    1 monumental


## Part 3: Creating Bag of Words representation

Define a dictionary that maps each vocab word to an integer defining its ordering in the vocabulary set

In [79]:
# For now we're not removing any words for the sake of using BoW size as a hyper parameter, so sorted_tokens is just entire tokens array

vocab_dict = dict()
for vocab_id, tok in enumerate(sorted_tokens):
    vocab_dict[tok] = vocab_id

## Part 4: Using CountVectorizer and Logistic Regression for Binary Classification

Import the library:

In [80]:
from sklearn.feature_extraction.text import CountVectorizer

### Create a Counter Vectorizer that uses our vocabulary list:

In [81]:
bow_preprocessor = CountVectorizer(binary=False, vocabulary=vocab_dict)

bow_preprocessor.fit(tr_text_list)

,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (strip_accents and lowercase) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None
,"stop_words stop_words: {'english'}, list, default=NoneIf 'english', a built-in stop word list for English is used.There are several known issues with 'english' and you shouldconsider an alternative (see :ref:`stop_words`).If a list, that list is assumed to contain stop words, all of whichwill be removed from the resulting tokens.Only applies if ``analyzer == 'word'``.If None, no stop words will be used. In this case, setting `max_df`to a higher value, such as in the range (0.7, 1.0), can automatically detectand filter stop words based on intra corpus document frequency of terms.",None
,"token_pattern token_pattern: str or None, default=r""(?u)\\b\\w\\w+\\b""Regular expression denoting what constitutes a ""token"", only usedif ``analyzer == 'word'``. The default regexp select tokens of 2or more alphanumeric characters (punctuation is completely ignoredand always treated as a token separator).If there is a capturing group in token_pattern then thecaptured group content, not the entire match, becomes the token.At most one capturing group is permitted.",'(?u)\\b\\w\\w+\\b'
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentword n-grams or char n-grams to be extracted. All values of n suchsuch that min_n <= n <= max_n will be used. For example an``ngram_range`` of ``(1, 1)`` means only unigrams, ``(1, 2)`` meansunigrams and bigrams, and ``(2, 2)`` means only bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"analyzer analyzer: {'word', 'char', 'char_wb'} or callable, default='word'Whether the feature should be made of word n-gram or charactern-grams.Option 'char_wb' creates character n-grams only from text insideword boundaries; n-grams at the edges of words are padded with space.If a callable is passed it is used to extract the sequence of featuresout of the raw, unprocessed input... versionchanged:: 0.21Since v0.21, if ``input`` is ``filename`` or ``file``, the data isfirst read from the file and then passed to the given callableanalyzer.",'word'


Then we use scikit `transform` to have a sparse matrix transformation to obtain the features:

In [82]:
sparse_arr = bow_preprocessor.transform(tr_text_list)
print(type(sparse_arr))
print(sparse_arr.shape)

# Convert to a dense representation

dense_arr_3V = sparse_arr.toarray()
print(type(dense_arr_3V))
print(dense_arr_3V.shape)

dense_arr_3V

<class 'scipy.sparse._csr.csr_matrix'>
(5557, 34884)
<class 'numpy.ndarray'>
(5557, 34884)


array([[3, 5, 3, ..., 0, 0, 0],
       [4, 0, 1, ..., 0, 0, 0],
       [6, 2, 2, ..., 0, 0, 0],
       ...,
       [4, 1, 1, ..., 0, 0, 0],
       [3, 3, 1, ..., 1, 1, 0],
       [6, 3, 1, ..., 0, 0, 1]], shape=(5557, 34884))

### CountVectorizer that builds its own UNIGRAM vocabulary

* ngram_range=(1,1) : Means it will use unigrams (individual tokens)
* min_df=1 : Means include any term that occurs in at least one document in training set
* max_df=1.0 : Means include any terms that appears in less than 100% (fraction of 1.0) of training docs
* binary=False : means it will do counts, not just binary presence/absence of each vocab term


In [83]:
bow_preprocessor = CountVectorizer(ngram_range=(1,1), min_df=1, max_df=1.0, binary=False)

In [84]:
bow_preprocessor.fit(tr_text_list)
len(bow_preprocessor.vocabulary_)

26866

Print first 10 words in vocabulary for testing

In [85]:
for term, count in list(bow_preprocessor.vocabulary_.items())[:10]:
    print("%4d %s" % (count, term))

26745 yes
26144 what
22259 sort
16713 of
23784 terms
25960 was
11530 he
16804 on
26418 with
23847 the


## Part 5: Bag of Words Count Vectorizer + Classifier Pipeline

### Create a pipeline + Classifier

In [87]:
import sklearn

In [88]:
my_bow_classifier_pipeline = sklearn.pipeline.Pipeline([
    ('my_bow_feature_extractor', CountVectorizer(min_df=1, max_df=1.0, ngram_range=(1,1))),
    ('my_classifier', sklearn.linear_model.LogisticRegression(C=1.0, max_iter=20, random_state=101)),
])

In [91]:
my_parameter_grid_by_name = dict()
my_parameter_grid_by_name['my_bow_feature_extractor__min_df'] = [1, 2, 4]
my_parameter_grid_by_name['my_classifier__C'] = np.logspace(-4, 4, 9)

my_scoring_metric_name = 'accuracy'

N

5557

In [93]:
print(y_train_df)

           author                  title  passage_id   Coarse Label  \
0     A. A. Milne  The Red House Mystery         618  Key Stage 2-3   
1     A. A. Milne  The Red House Mystery         685  Key Stage 2-3   
2     A. A. Milne  The Red House Mystery         833  Key Stage 2-3   
3     A. A. Milne  The Red House Mystery         959  Key Stage 2-3   
4     A. A. Milne  The Red House Mystery        1019  Key Stage 2-3   
...           ...                    ...         ...            ...   
5552   Émile Zola            L'Assommoir       11405  Key Stage 4-5   
5553   Émile Zola            L'Assommoir       11810  Key Stage 4-5   
5554   Émile Zola            L'Assommoir       13085  Key Stage 4-5   
5555   Émile Zola            L'Assommoir       13699  Key Stage 4-5   
5556   Émile Zola            L'Assommoir       14352  Key Stage 4-5   

             Fine Label  
0     Key Stage 2 (KS2)  
1     Key Stage 2 (KS2)  
2     Key Stage 2 (KS2)  
3     Key Stage 2 (KS2)  
4     Key Stage 2